# EX — Agents & Orchestration Real-World Exercise

Implement a minimal ReAct-style loop (Reason -> Act -> Observe) by hand, offline,
so you understand exactly what an agent framework automates for you.


In [ ]:
def get_order_status(order_id):
    fake_db = {"98765": "Shipped, arriving in 2 days", "12345": "Delivered"}
    return fake_db.get(order_id, "Order not found")

def initiate_refund(order_id):
    return f"Refund initiated for order {order_id}. Funds will arrive in 5-7 business days."

TOOLS = {
    "get_order_status": {
        "fn": get_order_status,
        "description": "Look up shipping status. Args: order_id (str)",
    },
    "initiate_refund": {
        "fn": initiate_refund,
        "description": "Start a refund for an order. Args: order_id (str)",
    },
}


## 1. A Mock 'Planner' That Picks a Tool
In a real agent, an LLM reads the user message + tool descriptions and decides what to call. Here we mock that decision with simple keyword rules so the loop logic is visible.

In [ ]:
import re

def mock_plan(user_message, history):
    """Very simplified stand-in for an LLM's reasoning step."""
    order_match = re.search(r"\b\d{5}\b", user_message)
    order_id = order_match.group() if order_match else None

    if "refund" in user_message.lower() or "money back" in user_message.lower():
        if order_id:
            return {"action": "initiate_refund", "args": {"order_id": order_id}}
        return {"action": "final_answer", "content": "Which order number is this for?"}
    if "status" in user_message.lower() or "track" in user_message.lower():
        if order_id:
            return {"action": "get_order_status", "args": {"order_id": order_id}}
        return {"action": "final_answer", "content": "Which order number would you like tracked?"}
    return {"action": "final_answer", "content": "I can help with order status or refunds — what do you need?"}


## 2. The Agent Loop

In [ ]:
def run_agent(user_message, max_steps=3):
    history = []
    for step in range(max_steps):
        plan = mock_plan(user_message, history)
        print(f"[step {step}] plan: {plan}")
        if plan["action"] == "final_answer":
            return plan["content"]
        tool = TOOLS.get(plan["action"])
        if tool is None:
            return f"Error: unknown tool {plan['action']}"
        observation = tool["fn"](**plan["args"])
        print(f"[step {step}] observation: {observation}")
        history.append((plan, observation))
        return observation  # in this simplified mock, one tool call is enough to answer
    return "Max steps reached without a final answer."

print(run_agent("What's the status of order 98765?"))


### TODO 1
Run the agent on: `'I want my money back for order 12345'` and on an ambiguous message with no order number, e.g. `'I need a refund'`. Confirm the agent asks a clarifying question instead of guessing.

In [ ]:
# TODO


<details><summary>Solution</summary>

```python
print(run_agent("I want my money back for order 12345"))
print(run_agent("I need a refund"))
```
</details>

## 3. Adding a Step Limit & Human Approval Gate
**Pointer:** never let a refund/purchase/send-email action fire without a human-approval step in a real system.

In [ ]:
def run_agent_with_approval(user_message, approve_fn, max_steps=3):
    plan = mock_plan(user_message, [])
    if plan["action"] == "initiate_refund":
        if not approve_fn(plan):
            return "Refund not approved by reviewer; no action taken."
    if plan["action"] == "final_answer":
        return plan["content"]
    tool = TOOLS[plan["action"]]
    return tool["fn"](**plan["args"])

def always_approve(plan):
    print("APPROVAL REQUEST:", plan)
    return True  # in a real system, a human reviews this before returning True

print(run_agent_with_approval("refund my order 98765", always_approve))


## Key Takeaways
- An agent loop is Reason (pick action) -> Act (call tool) -> Observe (read result) -> repeat until done.
- Always cap the number of steps — a confused agent can loop forever otherwise.
- Gate irreversible actions (refunds, purchases, sending messages) behind an explicit approval step.
- Clear tool descriptions + fallback clarifying questions prevent the agent from guessing on missing info.
